# DTA Provenance Standards Demo

**Interactive tutorial for Data & Trust Alliance provenance tracking**

This notebook demonstrates:
1. Creating DTA-compliant provenance metadata
2. Git-native provenance tracking
3. Real-world examples (Healthcare, ML, IoT, Financial)
4. Validation and verification
5. Audit trail generation

---

## Setup

First, let's import the necessary libraries:

In [ ]:
# Add git-native to path
import sys
from pathlib import Path

# Add the git-native src directory to Python path
git_native_path = Path.cwd() / 'git-native'
if str(git_native_path) not in sys.path:
    sys.path.insert(0, str(git_native_path))

# Import provenance tracking modules
from src.provenance import ProvenanceMetadata, ProvenanceTracker, load_provenance_file
from src.verify import validate_provenance_file

# Standard libraries
import json
import tempfile
from datetime import datetime

print("✅ Setup complete!")

## Part 1: Understanding DTA Standards

The Data & Trust Alliance defines **22 fields** in 3 categories:

### SOURCE (Where data came from)
- `datasetName` ⭐ Required
- `providerName` ⭐ Required
- `datasetVersion`
- `datasetURI`
- `providerWebsite`
- `geographicSourceOfData`
- `dataOriginCountry`
- `locationDataGenerated`

### PROVENANCE (How data was created)
- `dataGenerationMethod` ⭐ Required
- `dateDataGenerated` ⭐ Required  
- `dataType` ⭐ Required
- `dataFormat` ⭐ Required
- `dataSubjectivity`
- `qualityIndicators`

### USE (How data should be used)
- `intendedUse` ⭐ Required
- `legalRightsToUse` ⭐ Required
- `sensitiveData` ⭐ Required
- `restrictions`
- `sensitiveDataCategories` (required if sensitiveData=true)
- `privacyMeasures` (required if sensitiveData=true)
- `dataProcessingLocation`

## Part 2: Creating Provenance Metadata

Let's create a simple example for a customer churn dataset:

In [ ]:
# Create DTA-compliant metadata
metadata = ProvenanceMetadata(
    source={
        "datasetName": "Customer Churn Training Data",
        "providerName": "Marketing Analytics Team",
        "datasetVersion": "1.0.0",
        "providerWebsite": "https://example.com/analytics"
    },
    provenance={
        "dataGenerationMethod": "SQL export from production database with 90-day lookback",
        "dateDataGenerated": "2024-01-15T00:00:00Z",
        "dataType": "Tabular",
        "dataFormat": "CSV with 12 columns: customer_id, signup_date, tenure_months, monthly_charges, total_charges, contract_type, payment_method, internet_service, tech_support, online_security, device_protection, churn_label",
        "qualityIndicators": {
            "completeness": 0.98,
            "recordCount": 7043,
            "missingValueRate": 0.02
        }
    },
    use={
        "intendedUse": "Training ML model for customer churn prediction",
        "legalRightsToUse": "Internal business data - legitimate business interest per GDPR Article 6(1)(f)",
        "sensitiveData": True,
        "sensitiveDataCategories": ["PII", "Customer behavior data", "Financial information"],
        "privacyMeasures": "Customer IDs pseudonymized with SHA-256, names and addresses removed, geographic data aggregated to ZIP code level",
        "restrictions": "Internal use only. Do not share externally. Comply with data retention policy (delete after 2 years)."
    }
)

print("✅ Metadata created!")
print(f"\nMetadata hash: {metadata.compute_hash()[:16]}...")
print(f"\nMetadata preview:")
print(json.dumps(metadata.to_dict(), indent=2)[:500] + "...")

## Part 3: Git-Native Provenance Tracking

Let's create a demo repository and track provenance:

In [ ]:
# Create a temporary Git repository for demo
import git

# Create temp directory
demo_dir = Path(tempfile.mkdtemp(prefix="dta_demo_"))
print(f"📁 Created demo directory: {demo_dir}")

# Initialize Git repository
repo = git.Repo.init(demo_dir)
with repo.config_writer() as git_config:
    git_config.set_value("user", "name", "Demo User")
    git_config.set_value("user", "email", "demo@example.com")

# Create a sample dataset
dataset_file = demo_dir / "customer_churn.csv"
dataset_file.write_text(
    "customer_id,tenure_months,monthly_charges,churn_label\n"
    "c001,24,79.99,0\n"
    "c002,3,59.99,1\n"
    "c003,48,89.99,0\n"
    "c004,1,29.99,1\n"
    "c005,36,69.99,0\n"
)

print(f"📊 Created sample dataset with 5 records")
print(f"\n{dataset_file.read_text()[:200]}...")

### Commit Data with Provenance

In [ ]:
# Initialize provenance tracker
tracker = ProvenanceTracker(demo_dir)

# Commit with provenance
commit_hash = tracker.commit_with_provenance(
    file_paths=[dataset_file],
    metadata=metadata,
    message="Add customer churn training data v1.0",
    sign=False
)

print(f"✅ Committed with provenance!")
print(f"Commit hash: {commit_hash}")

# Show the commit
commit = repo.commit(commit_hash)
print(f"\nCommit message:\n{commit.message[:300]}...")

### Verify Integrity

In [ ]:
# Read provenance from commit
read_metadata = tracker.read_provenance(commit_hash)

if read_metadata:
    print("✅ Provenance metadata found in commit!")
    print(f"\nDataset: {read_metadata.source['datasetName']}")
    print(f"Provider: {read_metadata.source['providerName']}")
    print(f"Generated: {read_metadata.provenance['dateDataGenerated']}")
    print(f"Sensitive: {read_metadata.use['sensitiveData']}")
    
    # Verify integrity
    is_valid, message = tracker.verify_integrity(commit_hash)
    print(f"\n{'✅' if is_valid else '❌'} Integrity check: {message}")
else:
    print("❌ No provenance metadata found")

### Generate Audit Trail

In [ ]:
# Let's make some updates to show audit trail

# Update 1: Add more data
with open(dataset_file, 'a') as f:
    f.write("c006,12,49.99,0\nc007,6,39.99,1\n")

metadata_v2 = ProvenanceMetadata(
    source=metadata.source | {"datasetVersion": "1.1.0"},
    provenance=metadata.provenance | {
        "dateDataGenerated": "2024-02-01T00:00:00Z",
        "qualityIndicators": {"completeness": 0.99, "recordCount": 7}
    },
    use=metadata.use
)

commit_hash_v2 = tracker.commit_with_provenance(
    file_paths=[dataset_file],
    metadata=metadata_v2,
    message="Add 2 more customer records"
)

print(f"✅ Added version 1.1.0: {commit_hash_v2}")

# Update 2: Fix data quality issue
content = dataset_file.read_text()
content = content.replace("c004,1,29.99,1", "c004,1,29.99,0")  # Fix misclassification
dataset_file.write_text(content)

metadata_v3 = ProvenanceMetadata(
    source=metadata.source | {"datasetVersion": "1.2.0"},
    provenance=metadata.provenance | {
        "dateDataGenerated": "2024-02-05T00:00:00Z",
        "dataGenerationMethod": "Fixed misclassification in record c004",
        "qualityIndicators": {"completeness": 0.99, "recordCount": 7, "correctionsMade": 1}
    },
    use=metadata.use
)

commit_hash_v3 = tracker.commit_with_provenance(
    file_paths=[dataset_file],
    metadata=metadata_v3,
    message="Fix misclassification in record c004"
)

print(f"✅ Added version 1.2.0: {commit_hash_v3}")

In [ ]:
# Generate audit trail
audit_trail = tracker.generate_audit_trail(dataset_file, max_commits=10)

print("📋 Audit Trail for customer_churn.csv:\n")
print(f"{'='*80}")

for i, record in enumerate(audit_trail, 1):
    print(f"\n#{i} Commit: {record['commit_hash'][:8]}")
    print(f"   Date: {record['commit_date']}")
    print(f"   Message: {record['commit_message']}")
    print(f"   Author: {record['author']}")
    
    if record['provenance_metadata']:
        meta = record['provenance_metadata']
        print(f"   📊 Dataset: {meta.get('source', {}).get('datasetName', 'N/A')}")
        print(f"   📅 Generated: {meta.get('provenance', {}).get('dateDataGenerated', 'N/A')}")
        if 'qualityIndicators' in meta.get('provenance', {}):
            qi = meta['provenance']['qualityIndicators']
            print(f"   ✨ Quality: {qi}")
    print(f"   Hash: {record['provenance_hash'][:16]}..." if record['provenance_hash'] else "   (No provenance hash)")

print(f"\n{'='*80}")
print(f"✅ Total commits with provenance: {len([r for r in audit_trail if r['provenance_metadata']])}")

## Part 4: Real-World Examples

Let's load and validate the official examples from the project:

In [ ]:
# Load healthcare example
examples_dir = Path.cwd() / 'standards' / 'examples'

examples = {
    "Healthcare Imaging": "healthcare-imaging.json",
    "ML Training (HuggingFace)": "ml-training-huggingface.json",
    "IoT Sensor Stream": "iot-sensor-stream.json",
    "Financial Transactions": "financial-transactions.json"
}

for name, filename in examples.items():
    filepath = examples_dir / filename
    if filepath.exists():
        print(f"\n{'='*80}")
        print(f"📁 {name}")
        print(f"{'='*80}")
        
        # Load metadata
        metadata = load_provenance_file(filepath)
        
        print(f"✅ Dataset: {metadata.source['datasetName']}")
        print(f"   Provider: {metadata.source['providerName']}")
        print(f"   Type: {metadata.provenance['dataType']}")
        print(f"   Format: {metadata.provenance['dataFormat'][:80]}...")
        print(f"   Sensitive: {metadata.use['sensitiveData']}")
        if metadata.use['sensitiveData']:
            print(f"   Categories: {metadata.use.get('sensitiveDataCategories', [])}")
            print(f"   Privacy: {metadata.use.get('privacyMeasures', 'N/A')[:80]}...")
        
        # Validate
        report = validate_provenance_file(filepath)
        print(f"\n   {'✅ VALID' if report.is_valid else '❌ INVALID'} - Compliance Score: {report.score:.1%}")
        if report.errors:
            print(f"   Errors: {len(report.errors)}")
        if report.warnings:
            print(f"   Warnings: {len(report.warnings)}")
    else:
        print(f"⚠️  {name}: File not found at {filepath}")

## Part 5: Healthcare Example Deep Dive

Let's examine the healthcare example in detail:

In [ ]:
healthcare_file = examples_dir / "healthcare-imaging.json"

if healthcare_file.exists():
    with open(healthcare_file) as f:
        healthcare_data = json.load(f)
    
    print("🏥 Healthcare Imaging Dataset - DTA Compliance\n")
    print("="*80)
    
    # SOURCE
    print("\n📍 SOURCE (Where the data came from)")
    print("-" * 40)
    for key, value in healthcare_data.get('source', {}).items():
        print(f"  {key}: {value}")
    
    # PROVENANCE
    print("\n🔬 PROVENANCE (How the data was created)")
    print("-" * 40)
    for key, value in healthcare_data.get('provenance', {}).items():
        if isinstance(value, dict):
            print(f"  {key}:")
            for k, v in value.items():
                print(f"    - {k}: {v}")
        elif isinstance(value, list):
            print(f"  {key}: {len(value)} items")
        else:
            print(f"  {key}: {str(value)[:100]}")
    
    # USE
    print("\n⚖️  USE (How the data should be used)")
    print("-" * 40)
    for key, value in healthcare_data.get('use', {}).items():
        if isinstance(value, list):
            print(f"  {key}:")
            for item in value:
                print(f"    - {item}")
        else:
            display_value = str(value)[:200]
            if len(str(value)) > 200:
                display_value += "..."
            print(f"  {key}: {display_value}")
    
    print("\n" + "="*80)
    
    # Key highlights
    print("\n🔑 Key Highlights:")
    print("  • HIPAA-compliant de-identification")
    print("  • IRB approval documented")
    print("  • Quality metrics included")
    print("  • Inter-rater agreement tracked")
    print("  • Equipment specifications documented")
else:
    print("Healthcare example not found")

## Part 6: Summary and Best Practices

### ✅ Key Takeaways

1. **Required Fields** - Always include:
   - datasetName, providerName
   - dataGenerationMethod, dateDataGenerated, dataType, dataFormat
   - intendedUse, legalRightsToUse, sensitiveData

2. **Sensitive Data** - If sensitiveData=true:
   - Must include sensitiveDataCategories
   - Must include privacyMeasures

3. **Version Tracking** - Use datasetVersion for lineage

4. **Quality Indicators** - Document data quality metrics

5. **Legal Basis** - Clearly state legal rights (GDPR, licenses, etc.)

### 🎯 When to Use Git-Native vs Blockchain

**Use Git-Native (this demo) when:**
- Single organization
- Internal ML pipelines
- Privacy-sensitive data
- High-frequency updates
- Free solution needed

**Use Blockchain when:**
- Multiple untrusting parties
- Public transparency needed
- No central authority
- Automated settlement required

### 📚 Learn More

- [DTA Standards Documentation](../docs/DTA_STANDARDS.md)
- [Architecture Guide](../docs/ARCHITECTURE.md)
- [Comparison: Git vs Blockchain](../docs/COMPARISON.md)
- [Official DTA Website](https://www.dtaalliance.org/)

---

**Questions? Issues? Contributions?**
- GitHub: [https://github.com/yourusername/dta-provenance-demo](https://github.com/yourusername/dta-provenance-demo)
- Documentation: See the `docs/` directory

🎉 **Thanks for trying the DTA Provenance Standards Demo!**

In [ ]:
# Cleanup demo repository
import shutil

try:
    shutil.rmtree(demo_dir)
    print(f"🧹 Cleaned up demo directory: {demo_dir}")
except:
    print(f"⚠️  Could not clean up demo directory: {demo_dir}")
    print("   You may want to delete it manually.")